# Prepare Solar Panel Dataset from New Panel Delineation

* Calls all panel shapes existing within all existing and digitized array boundaries (bound by NAIP availability)
* Checks for panel delination quality and resulting array shape quality, removes low quality panel delineations based on perimeter to area ratio
    * USPVDB and CCVPV have both been manually validated for correctness in their original creation
    * Saves arrays (excluding USPVDB and CCVPV) where panels are not present OR panel delination was low quality, and exports as shape file to assess for comissions
* Removes arrays and panels manually validated as comissions from remote sensing datasets
* Creates new array shapes by buffering and dissolving panel boundaries
    * USPVDB and CCVPV are both high array quality. So keep those shapes in the array dataset, but improve all other array boundaries if the new shape is of high quality
* Saves high-quality panels objects as new shape file
* Saves highest-quality array objects as new shape file

## Import Libraries and Variables

In [12]:
# Import libraries
import numpy as np
import pandas as pd
import geopandas as gpd
import os 

# Import gmseusUtils
import gmseusUtils as gu

In [13]:
# Set folder paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
derivedTemp_path = os.path.join(derived_path, r'intermediateProducts')

# Set getPanels from GEE output folder path for two different approaches
getPanels_path = os.path.join(derivedTemp_path, r'getPanelsGEEOutput')

# GM-SEUS output initial paths
gmseusArraysInitPath = os.path.join(derivedTemp_path, r'existingDatasetArrayShapes.shp')
gmseusPanelsInitPath = os.path.join(derivedTemp_path, r'existingDatasetPanelShapes.shp')

# GM-SEUS prior version NAIP panels path
gmseusNaipPanelsPriorVersionPath = os.path.join(downloaded_path, r'SolarDB\GMSEUS_v1_0\SHP\GMSEUS_NAIP_Panels.shp')

# Set GM-SEUS NAIP classified panels and arrays paths and no QAQC panels path
gmseusNaipArraysPath = os.path.join(derivedTemp_path, r'GMSEUS_NAIP_Arrays.shp')
gmseusNaipPanelsPath = os.path.join(derivedTemp_path, r'GMSEUS_NAIP_Panels.shp')
gmesusNaipPanelsNoQAQCPath = os.path.join(derivedTemp_path, r'GMSEUS_NAIP_PanelsNoQAQC.shp')

# Load the config from the text file
config = gu.load_config('config.txt')

# Set general variables
currentVersion = config['currentVersion'] # version of the dataset
gee_crs = config['gee_crs'] # native projection of Google Earth Engine exports
minPanelRowArea = config['minPanelRowArea'] # 15 m2, minimum area for a single panel row from the 1st percentile panel area from Stid et al., 2022
maxPanelRowArea = config['maxPanelRowArea'] # 254 m2 95th perccentile for a single panel row from Stid et al., 2022. MSU Solar Carport has max 1890m2
minNumPanelRows = config['minNumPanelRows'] # 3 panels, minimum number of panels rows to form a ground mounted solar array, definition from Stid et al., 2022
minPmArRatio = config['minPmArRatio'] # 18.8%, 20% was minimum ratio of panel perimeter to area ratio for panels from Stid et al., 2022, MSU Solar Carport has min 18.9%
panelArrayBuff = config['panelArrayBuff'] # 10m buffer, 20m maximum distance between panel rows to form an array. We used 5m in Stid et al., 2022, but there are lower packing factors at greater latitudes (nativeID: '1229957948')
arrayArrayBuff = config['arrayArrayBuff'] # 20m buffer, 40m maximum distance between arrays subsections of the same mount type to form a complete array. In Stid et al., 2022, we used 50m, but we checked for same installation year in addition to mount type.

# Set limits for mount classification
lengthRatioThresh = config['lengthRatioThresh']  # If length ratio < 3.0, set to dual_axis or else fixed_axis_diagonal, else single- or fixed-axis
areaRatioThresh = config['areaRatioThresh']  # If area ratio < 0.15, set to fixed_diag_axis, else dual_axis
toCRS = config['to_crs']  # EPSG:6350 NAD83 (2011)

# Append toCRS with the EPSG prefix for use in GeoPandas
toCRS = f'EPSG:{toCRS}'

# Set the threshold for Z-scores (3 standard deviations is a common choice, adjust if needed) and unique mount proportion. 
z_threshold = 3 # 3 standard deviations
uniqueMountThreshold = 0.1 # 10% of the panel mount types in the array

# Remove new arrays where the new array area is less than 0.25 * gmseus array area and where new array PmArRatio is less than the 99th percentile of gmseus PmArRatio
newAreaThreshold = 0.25 # 25% of the original gmseus array area
gmseusPmArRatioThreshold = 0.99 # 99th percentile

# For this script where we filter panel-rows based on shared geometrical similarity, the arrayID column is subArrID, because we exlode all sub array geometries in script2
arrayIDcol = 'subArrID'

## Process Panel Data from GEE Outputs

In [6]:
# Get the solar panels geodataframe
solarPanels = gu.getPanels_method(getPanels_path)

# Print the number of panels in each geodataframe and the total area of solar panels
print(f'Total number of newly delineated panel-rows: {len(solarPanels)}')
print(f'Total area of solar panels on initial processing: {solarPanels.geometry.area.sum() / 1e6:.2f} km2')

# Export the solar panels geodataframes
solarPanels.to_file(gmesusNaipPanelsNoQAQCPath, driver='ESRI Shapefile')

Both geojson and shapefile found in the folder. Concatenating both.
Total number of newly delineated panel-rows: 2516161
Total area of solar panels on initial processing: 719.84 km2


## Filter for High Quality Panels and Create New Array Dataset

### Remove inividual panels by within array by design/shape similarity

In [7]:
# ~600 minutes 

# Call solar panels
# solarPanels = gpd.read_file(gmesusNaipPanelsNoQAQCPath)

# Get the mount type, azimuth, length ratio, area ratio, short edge, and long edge for each panel
solarPanels[['mount', 'azimuth', 'lengthRatio', 'shortEdge', 'longEdge']] = solarPanels.apply(gu.assignMountType, axis=1, result_type='expand')

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Create initial solar arrays from solar panels, and remove low quality arrays and panels

# First, drop any panel below the minimum panel area, and greater than the max panel area
solarPanels = solarPanels[solarPanels.geometry.area > minPanelRowArea]
solarPanels = solarPanels[solarPanels.geometry.area < maxPanelRowArea]

# Save solarArrays as copy of solarPanels
solarPanels = solarPanels.reset_index(drop=True)

# Set an initial panelID 1 through n for the entire dataset
solarPanels['panelID'] = range(1, len(solarPanels) + 1)

# Create solar arrays from solar panels
solarArrays = gu.createArrayFromPanels(solarPanels, panelArrayBuff, arrayIDcol)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Determine panel quality based on panel design parameters within arrays

'''
Here, we will remove commissions based on a number of quality control metrics and determining if panels are outliers. 
We consider 1 universal metric and 5 within array (local) metrics: PmArRatio (universal), PmArRatio (local), lengthRatio, areaRatio, compactness, and mount proportion. 
First, we will determine if a panel is universally an outlier with perimeter to area ratio (PmArRatio) using a set threshold, then within arrays using mount technology proportions, PmArRatio, lengthRatio, areaRatio, and compactness (Poslby-Popper ratio). 
If 2 or more fail, we will remove the panel.

PmArRatio is well contrained for square to rectangular objects of solar panel-row size (see percentiles of gmseusInitPanels).
We will set a universal miminum threshold for PmArRatio across the entire array to remove large area panel objects that are not solar panels. 
In gmseusInit, 0.19 was the minimum PmArRatio. We will use 0.18 as the minimum threshold for solar panels.
Universally, low compactness can include long skinny panel-rows and high compactness can include-dual axis or square panel-rows. 
So we will only use this metric to address within-array varaibility.
Within each array, we will also remove panel-objects where the mount type composes less than 10% of the array. 
'''

# Set failure threshold (max would be 5)
failureThreshold = 2

# Set an QAQC column as zero, which we will append to for each failed quality control metric. 
solarPanels['QAQC'] = 0

# Calculate the perimeter to area ratio and compacntess for each panel
solarPanels['perimeter'] = solarPanels.geometry.length
solarPanels['PmArRatio'] = solarPanels['perimeter'] / solarPanels['area']
solarPanels['compactness'] = (4 * np.pi * solarPanels['area']) / (solarPanels['perimeter'] ** 2)

# Calculate area ratio, the calculate the panel area to bounding box area ratio, not the minimum bounding rectangle
solarPanels['bboxArea'] = solarPanels.geometry.bounds.apply(lambda row: (row['maxx'] - row['minx']) * (row['maxy'] - row['miny']), axis=1)
solarPanels['areaRatio'] = solarPanels['area'] / solarPanels['bboxArea']
solarPanels = solarPanels.drop(columns='bboxArea')

# First, universal removal of comissions based on constrained geometries of ground mounted solar panel-rows (PmArRatio). If PmArRatio is less than the minimum threshold, remove the panel.
solarPanels = solarPanels[solarPanels['PmArRatio'] >= minPmArRatio]

# Second, remove local (within array) comissions based on within-array mount similarity. If the mount type composes less than 10% of the array, add 1 to QAQC column.
solarPanels['uniqueMount'] = solarPanels.groupby([arrayIDcol, 'mount'])['mount'].transform('count') / solarPanels.groupby(arrayIDcol)['mount'].transform('count')
solarPanels.loc[solarPanels['uniqueMount'] < uniqueMountThreshold, 'QAQC'] += 1
solarPanels = solarPanels.drop(columns='uniqueMount')

# Third, remove local (within array) comissions based on within-array PmArRatio, lengthRatio, areaRatio, and compactness (Poslby-Popper ratio). If the panel is an outlier in any of these metrics, add 1 to QAQC column.
solarPanels['PmArRatioZ'] = (solarPanels['PmArRatio'] - solarPanels.groupby(arrayIDcol)['PmArRatio'].transform('mean')) / solarPanels.groupby(arrayIDcol)['PmArRatio'].transform('std')
solarPanels['lengthRatioZ'] = (solarPanels['lengthRatio'] - solarPanels.groupby(arrayIDcol)['lengthRatio'].transform('mean')) / solarPanels.groupby(arrayIDcol)['lengthRatio'].transform('std')
solarPanels['areaRatioZ'] = (solarPanels['areaRatio'] - solarPanels.groupby(arrayIDcol)['areaRatio'].transform('mean')) / solarPanels.groupby(arrayIDcol)['areaRatio'].transform('std')
solarPanels['compactnessZ'] = (solarPanels['compactness'] - solarPanels.groupby(arrayIDcol)['compactness'].transform('mean')) / solarPanels.groupby(arrayIDcol)['compactness'].transform('std')
solarPanels.loc[solarPanels['PmArRatioZ'].abs() > z_threshold, 'QAQC'] += 1
solarPanels.loc[solarPanels['lengthRatioZ'].abs() > z_threshold, 'QAQC'] += 1
solarPanels.loc[solarPanels['areaRatioZ'].abs() > z_threshold, 'QAQC'] += 1
solarPanels.loc[solarPanels['compactnessZ'].abs() > z_threshold, 'QAQC'] += 1

# Drop the Z-score columns
solarPanels = solarPanels.drop(columns=['PmArRatioZ', 'lengthRatioZ', 'areaRatioZ', 'compactnessZ'])

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Remove low quality arrays and panels

# Get solar panels that fail three or more quality control metrics
solarPanelsDrop = solarPanels[solarPanels['QAQC'] >= failureThreshold]

# Get solar panels that pass quality control metrics
solarPanels = solarPanels[solarPanels['QAQC'] < failureThreshold]

# If the resulting array has three or fewer panels, add these to solarPanelsDrop and remove them from solarPanels
panelCounts = solarPanels.groupby(arrayIDcol).size().reset_index(name='numPanels')
solarPanels = solarPanels.merge(panelCounts, on=arrayIDcol, how='left')
solarPanels = solarPanels.reset_index(drop=True)

# Get the arrays with too few panels
solarPanelsTooFew = solarPanels[solarPanels['numPanels'] <= minNumPanelRows]
solarPanelsTooFew = solarPanelsTooFew.drop(columns='numPanels')
solarPanelsDrop = pd.concat([solarPanelsDrop, solarPanelsTooFew], ignore_index=True)
solarPanels = solarPanels[solarPanels['numPanels'] > minNumPanelRows]

# Export 
#solarPanelsDrop.to_file(os.path.join(derivedTemp_path, 'solarPanelsDrop.shp'), driver='ESRI Shapefile')

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Regenerate solar arrays from the filtered solar panels and export

# Regenerate solar arrays from the filtered solar panels
solarPanels = solarPanels.reset_index(drop=True)
solarPanels = solarPanels.drop(columns=['numPanels'])
solarArrays = gu.createArrayFromPanels(solarPanels, panelArrayBuff, arrayIDcol)

# Set a new arrayID and panelID that is 1 through n for the entire dataset. First drop old columns.
solarArrays = solarArrays.reset_index(drop=True)
solarPanels = solarPanels.reset_index(drop=True)
solarArrays = solarArrays.drop(columns=['panelID'])
solarPanels = solarPanels.drop(columns=[arrayIDcol, 'panelID'])
solarPanels['panelID'] = range(1, len(solarPanels) + 1)

# Get arrayID for each panel from a spatial join. Get only the arrayID column
solarPanels = gpd.sjoin(solarPanels, solarArrays[[arrayIDcol, 'geometry']], how='left', predicate='intersects')
solarPanels = solarPanels.drop(columns='index_right')

# Reset the index
solarPanels = solarPanels.reset_index(drop=True)
solarArrays = solarArrays.reset_index(drop=True)

# Export high quality arrays and panels
solarArrays.to_file(os.path.join(derivedTemp_path, 'solarArrays_ArrayQAQC.shp'), driver='ESRI Shapefile')
solarPanels.to_file(os.path.join(derivedTemp_path, 'solarPanels_PanelQAQC.shp'), driver='ESRI Shapefile')

# Export solar panels that are dropped
solarPanelsDrop.to_file(os.path.join(derivedTemp_path, 'solarPanelsDropped_PanelQAQC.shp'), driver='ESRI Shapefile')

# Print the number of panels dropped due to quality control within array
print(f'Total number of panels dropped due to quality control within arrays: {len(solarPanelsDrop)}')

C:\Users\stidjaco\AppData\Local\Temp\ipykernel_23844\2404535649.py:122: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  solarArrays.to_file(os.path.join(derivedTemp_path, 'solarArrays_ArrayQAQC.shp'), driver='ESRI Shapefile')
C:\Users\stidjaco\AppData\Local\Temp\ipykernel_23844\2404535649.py:123: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  solarPanels.to_file(os.path.join(derivedTemp_path, 'solarPanels_PanelQAQC.shp'), driver='ESRI Shapefile')
C:\Users\stidjaco\AppData\Local\Temp\ipykernel_23844\2404535649.py:126: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  solarPanelsDrop.to_file(os.path.join(derivedTemp_path, 'solarPanelsDropped_PanelQAQC.shp'), driver='ESRI Shapefile')


Total number of panels dropped due to quality control within arrays: 39333


### Remove array-wide panels by quality of new array delineation (new area compared to initial area, and perimeter to area ratio)
Importantly, we consider subarray shapes as an array, allowing for unique interarray design distinctions. Thus when we compare the new array area and PmArRatio to the initial, we must explode the initial arrays (this logic is already built into our new solar array delination)

In [8]:
# # Call solar panels and arrays
solarPanels = gpd.read_file(os.path.join(derivedTemp_path, 'solarPanels_PanelQAQC.shp'))
solarArrays = gpd.read_file(os.path.join(derivedTemp_path, 'solarArrays_ArrayQAQC.shp'))

# Call initial gmseus arrays
gmseusArraysInit = gpd.read_file(gmseusArraysInitPath)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ QAQC (Export high quality arrays and panels compared to initial GM-SEUS arrays)

# Reset the index, and calculate the area of the solar arrays
solarArrays = solarArrays.reset_index()
solarArrays['arrayArea'] = solarArrays.geometry.area

# Calculate the perimeter to area ratio of array 
solarArrays['PmArRatio'] = solarArrays.length / solarArrays.area

# Calculate array area and PmArRatio for gmseus initial arrays. First, explode initial gmseus arrays to match logic of panel-row generated arrays (dependent on sub-array shapes, allows for unique array designs within an array area).
gmseusArraysInit = gmseusArraysInit.explode(index_parts=False)
gmseusArraysInit = gmseusArraysInit.reset_index(drop=True)
gmseusArraysInit['arrayArea_gmseus'] = gmseusArraysInit.geometry.area
gmseusArraysInit['PmArRatio_gmseus'] = gmseusArraysInit.length / gmseusArraysInit.area

# Perform a spatial join to get the gmseus array area and PmArRatio
solarArrays = gpd.sjoin(solarArrays, gmseusArraysInit[['arrayArea_gmseus', 'PmArRatio_gmseus', 'geometry']], how='left', predicate='intersects')
solarArrays = solarArrays.drop(columns='index_right')

# Remove new arrays where the new array area is less than 0.25 * gmseus array area and where new array PmArRatio is less than the 99th percentile of gmseus PmArRatio
solarArraysHighQuality = solarArrays[(solarArrays['arrayArea'] > newAreaThreshold * solarArrays['arrayArea_gmseus']) & (solarArrays['PmArRatio'] < gmseusArraysInit['PmArRatio_gmseus'].quantile(gmseusPmArRatioThreshold))]
solarArraysHighQuality = solarArraysHighQuality.drop(columns=['arrayArea_gmseus', 'PmArRatio_gmseus'])
solarArraysHighQuality = solarArraysHighQuality.reset_index(drop=True)

# Drop level_0 and index columns if they exist
solarArraysHighQuality = solarArraysHighQuality.drop(columns=['level_0', 'index'], errors='ignore')

# Reset index, and filter for panels that are in high quality arrays
solarPanels = solarPanels.reset_index(drop=True)
solarPanelsHighQuality = solarPanels[solarPanels[arrayIDcol].isin(solarArraysHighQuality[arrayIDcol])].reset_index(drop=True)

# Drop level_0 and index columns if they exist
solarPanelsHighQuality = solarPanelsHighQuality.drop(columns=['level_0', 'index'], errors='ignore')

# Set Source for high quality panels and arrays Source = 'GMSEUS_' + currentVersion
solarArraysHighQuality['Source'] = 'GMSEUS_' + currentVersion
solarPanelsHighQuality['Source'] = 'GMSEUS_' + currentVersion

# Export high quality arrays and panels
solarArraysHighQuality.to_file(os.path.join(gmseusNaipArraysPath), driver='ESRI Shapefile')
solarPanelsHighQuality.to_file(os.path.join(gmseusNaipPanelsPath), driver='ESRI Shapefile')

# Print the number of panels removed due to array delineation quality control
print(f'Total number of panel-rows removed due to array delineation quality control: {len(solarPanels) - len(solarPanelsHighQuality)}')

# Print the final number of arrays and panels
print(f'Total number of solar arrays: {len(solarArraysHighQuality)}')
print(f'Total number of solar panel-rows: {len(solarPanelsHighQuality)}')

# Print the final area in km2 of arrays and panels
print(f'Total area of solar arrays: {solarArraysHighQuality.geometry.area.sum() / 1e6:.2f} km2')
print(f'Total area of solar panel-rows: {solarPanelsHighQuality.geometry.area.sum() / 1e6:.2f} km2')

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Export low quality arrays and panels to understand the quality of the delineation

# Reset all the indices
solarArrays = solarArrays.reset_index()
solarPanels = solarPanels.reset_index()
solarArraysHighQuality = solarArraysHighQuality.reset_index()
solarPanelsHighQuality = solarPanelsHighQuality.reset_index()

# Export low quality arrays and panels (not in solarArraysQAQC nativeID)
solarArraysLowQuality = solarArrays[~solarArrays[arrayIDcol].isin(solarArraysHighQuality[arrayIDcol])]
solarPanelsLowQuality = solarPanels[~solarPanels[arrayIDcol].isin(solarArraysHighQuality[arrayIDcol])]

# Export low quality arrays and panels
solarArraysLowQuality.to_file(os.path.join(derivedTemp_path, 'solarArraysLowArrayQuality.shp'), driver='ESRI Shapefile')
solarPanelsLowQuality.to_file(os.path.join(derivedTemp_path, 'solarPanelsLowArrayQuality.shp'), driver='ESRI Shapefile')

C:\Users\stidjaco\AppData\Local\Temp\ipykernel_23844\1057718605.py:43: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  solarArraysHighQuality.to_file(os.path.join(gmseusNaipArraysPath), driver='ESRI Shapefile')
C:\Users\stidjaco\AppData\Local\Temp\ipykernel_23844\1057718605.py:44: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  solarPanelsHighQuality.to_file(os.path.join(gmseusNaipPanelsPath), driver='ESRI Shapefile')


Total number of panel-rows removed due to array delineation quality control: 47374
Total number of solar arrays: 12225
Total number of solar panel-rows: 2410429
Total area of solar arrays: 907.49 km2
Total area of solar panel-rows: 417.88 km2


C:\Users\stidjaco\AppData\Local\Temp\ipykernel_23844\1057718605.py:70: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  solarArraysLowQuality.to_file(os.path.join(derivedTemp_path, 'solarArraysLowArrayQuality.shp'), driver='ESRI Shapefile')
C:\Users\stidjaco\AppData\Local\Temp\ipykernel_23844\1057718605.py:71: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  solarPanelsLowQuality.to_file(os.path.join(derivedTemp_path, 'solarPanelsLowArrayQuality.shp'), driver='ESRI Shapefile')


# Create the Highest Quality Panel Shapefile

## Merge New Panel Shapes with Existing Shapes

In [ ]:
# Call in GM-SEUS initial panels and GM-SEUS NAIP panels
gdf1 = gpd.read_file(gmseusPanelsInitPath)
gdf2 = gpd.read_file(gmseusNaipPanelsPath)
gdf3 = gpd.read_file(gmseusNaipPanelsPriorVersionPath)

# Save a gdf list
gdfList = [gdf1, gdf2, gdf3]

# Function to give a gdf list that is in preferential order for spatial filtering, and create merged panel datasets
def preferentialSpatialPanelRowFilter(gdfList, panelArrayBuff, arrayIDcol):

# Check that both GeoDataFrames have the toCRS
for i in range(len(gdfList)):
    if gdfList[i].crs != toCRS:
        gdfList[i] = gdfList[i].to_crs(toCRS)

# Set desired panel columns prior to merging
panelColumns = ['nativeID', 'Source', 'area', 'geometry']
for i in range(len(gdfList)):
    gdfList[i] = gdfList[i][panelColumns].copy()

# Create arrays from panels for each GeoDataFrame
arrayGdfList = []
for i in range(len(gdfList)):
    arrayGdf = gu.createArrayFromPanels(gdfList[i], panelArrayBuff, arrayIDcol)
    arrayGdfList.append(arrayGdf)

# Set source columns WITHIN GDF to compare after preferential spatial filtering
# pnlSource = gdf_1 for the first GeoDataFrame in the list, and so on
for i in range(len(gdfList)):
    gdfList[i]['pnlSource'] = f'gdf_{i+1}'
for i in range(len(arrayGdfList)):
    arrayGdfList[i]['pnlArrSource'] = f'gdf_{i+1}'

# Perform preferential spatial filtering on the array gdfs
prefArrays = gu.preferentialSpatialFilter(arrayGdfList)

# Perform spatial join for each original panel GDF → preferred arrays
# Keep only panels whose pnlSource matches pnlArrSource
filteredPanelsList = []
for i in range(len(gdfList)):
    joined = gpd.sjoin(
        gdfList[i],
        prefArrays[['pnlArrSource', 'geometry']],
        how='left',
        predicate='intersects'
    )

    # Keep only rows where the panel's source matches the chosen array source
    keep = joined[joined['pnlSource'] == joined['pnlArrSource']].copy()

    # Drop sjoin artifacts and pnlArrSource (optionally keep pnlSource)
    keep = keep.drop(columns=['index_right', 'pnlArrSource'], errors='ignore')

    filteredPanelsList.append(keep)

# Merge all preferentially filtered panels into a single GeoDataFrame
mergedPanels = pd.concat(filteredPanelsList, ignore_index=True)
mergedPanels = gpd.GeoDataFrame(mergedPanels, geometry='geometry', crs=toCRS)
mergedPanels = mergedPanels.reset_index(drop=True)

# Set the final panelID as 1 through n for the entire dataset
mergedPanels['panelID'] = range(1, len(mergedPanels) + 1)

In [9]:
# ~100 minutes

# Call in GM-SEUS initial panels and GM-SEUS NAIP panels
gmseusPanelsInit = gpd.read_file(gmseusPanelsInitPath)
gmseusNaipPanels = gpd.read_file(gmseusNaipPanelsPath)

# Set desired panel columns prior to merging
panelColumnsInit = ['nativeID', 'Source', 'area', 'geometry']

# Select desired columns from gmseusPanelsInit and gmseusNaipPanels
gmseusPanelsInit = gmseusPanelsInit[panelColumnsInit]
gmseusNaipPanels = gmseusNaipPanels[panelColumnsInit]

# Create arrays from panels for three datasets
gmseusPanelsInit_arrays = gu.createArrayFromPanels(gmseusPanelsInit, panelArrayBuff, '', '', False)
gmseusNaipPanels_arrays = gu.createArrayFromPanels(gmseusNaipPanels, panelArrayBuff, '', '', False)

# Set source columns to compare after preferential spatial filtering
gmseusPanelsInit['pnlSource'] = 'existing'
gmseusNaipPanels['pnlSource'] = 'naipNew'
gmseusPanelsInit_arrays['pnlArrSource'] = 'existing'
gmseusNaipPanels_arrays['pnlArrSource'] = 'naipNew'

# Perform preferential spatial filtering on the arrays created from panels
panelsArrayPriorityList = [gmseusPanelsInit_arrays, gmseusNaipPanels_arrays]
panelsArrayPriority = gu.preferentialSpatialFilter(panelsArrayPriorityList)

# Perform spatial join to get arraySource column attached to each panel-row dataset. Then, if panels source is equal to arraySource, keep the row. This will ensure that only panels that belong to the preferentially selected arrays are kept.
gmseusPanelsInit_arraysJoined = gpd.sjoin(gmseusPanelsInit, panelsArrayPriority[['pnlArrSource', 'geometry']], how='left', predicate='intersects')
gmseusNaipPanels_arraysJoined = gpd.sjoin(gmseusNaipPanels, panelsArrayPriority[['pnlArrSource', 'geometry']], how='left', predicate='intersects')
gmseusPanelsInit_filtered = gmseusPanelsInit_arraysJoined[gmseusPanelsInit_arraysJoined['pnlSource'] == gmseusPanelsInit_arraysJoined['pnlArrSource']]
gmseusNaipPanels_filtered = gmseusNaipPanels_arraysJoined[gmseusNaipPanels_arraysJoined['pnlSource'] == gmseusNaipPanels_arraysJoined['pnlArrSource']]

# Combine the panel datasets into a single geodataframe and clean
mergedPanels = pd.concat([gmseusPanelsInit_filtered, gmseusNaipPanels_filtered], ignore_index=True)
mergedPanels = gpd.GeoDataFrame(mergedPanels, geometry='geometry', crs=toCRS)
mergedPanels = mergedPanels.drop(columns=['pnlArrSource', 'index_right'], errors='ignore')
mergedPanels = mergedPanels.reset_index(drop=True)

# Set the final panelID as 1 through n for the entire dataset
mergedPanels['panelID'] = range(1, len(mergedPanels) + 1)

# If pnlSource is 'naipNew' set Source to 'GMSEUS' + 'v' + version. Else, maintain the Source column
mergedPanels.loc[mergedPanels['pnlSource'] == 'naipNew', 'Source'] = 'GMSEUS_' + currentVersion

# Drop pnlSource column
mergedPanels = mergedPanels.drop(columns='pnlSource')

# Print number of rows in mergedPanels
print(f'Number of final panel-rows in GM-SEUS panel-row dataset: {len(mergedPanels)}')

# Print the value counts of the Source column
print('Panel-row source value counts:')
print(mergedPanels['Source'].value_counts())

# Print the total sum of 'area' in the mergedPanels dataset in km2
print(f'Total area of panels in GM-SEUS panel-row dataset is {mergedPanels["area"].sum() / 1e6} km2')

# Export the mergedPanels dataset
mergedPanels.to_file(os.path.join(derivedTemp_path, r'GMSEUS_Panels_ExistingAndNAIP.shp'), driver='ESRI Shapefile')

Number of final panel-rows in GM-SEUS panel-row dataset: 3452409
Panel-row source value counts:
Source
GMSEUS_v2_0                   1925433
OSM                           1300096
CCVPV                          203330
GMSEUSdigArraysPanels_v2_0      23550
Name: count, dtype: int64
Total area of panels in GM-SEUS panel-row dataset is 518.2753002100429 km2


## Create Arrays from Panels for All New and Existing Panel-Rows

In [10]:
# Call in gmseusPanelsFinal
gmseusPanelsFinal = gpd.read_file(os.path.join(derivedTemp_path, r'GMSEUS_Panels_ExistingAndNAIP.shp'))

# Call in GM-SEUS initial arrays. Explode the arrays to match the logic of panel-row generated arrays (dependent on sub-array shapes, allows for unique array designs within an array area).
gmseusArraysInit = gpd.read_file(gmseusArraysInitPath)
gmseusArraysInit = gmseusArraysInit.explode(index_parts=False)
gmseusArraysInit = gmseusArraysInit.reset_index(drop=True)

# Add a column to gmseusArraysInit that called arrayIDcol (variable set at top of script) and is 1 through n for the entire dataset
gmseusArraysInit[arrayIDcol] = range(1, len(gmseusArraysInit) + 1) 

# Spatially join gmseus arrays to panels, copy the arrayID to the panels, and drop the index columns. 
gmseusPanelsFinal = gpd.sjoin(gmseusPanelsFinal, gmseusArraysInit[[arrayIDcol, 'geometry']], how='left', predicate='intersects')
gmseusPanelsFinal  = gmseusPanelsFinal.reset_index(drop=True)
gmseusPanelsFinal  = gmseusPanelsFinal.drop(columns=['index_left', 'index_right'], errors='ignore')
gmseusPanelsFinal = gmseusPanelsFinal.dropna(subset=[arrayIDcol]) # Redundent, drop panels that do not have an arrayID, in theory should never be the case

# Create arrays from panels -- despite the title, these arrays are generated from new naip panels and existing panels
gmseusArraysFromPanels = gu.createArrayFromPanels(gmseusPanelsFinal, panelArrayBuff, arrayIDcol)

# Print the number of arrays with panel-row-level infomraiton
print(f'Total number of arrays with panel-row-level information: {len(gmseusArraysFromPanels)}')

# Print the total area of gmseus arrays final
print(f'Total area of GM-SEUS arrays: {gmseusArraysFromPanels.geometry.area.sum() / 1e6:.2f} km2')

# Export the GM-SEUS Naip Arrays and gmseusArraysInit with arrayIDcol
gmseusArraysFromPanels.to_file(os.path.join(derivedTemp_path, r'GMSEUS_ArraysFromPanels.shp'), driver='ESRI Shapefile')
gmseusArraysInit.to_file(gmseusArraysInitPath, driver='ESRI Shapefile')

Total number of arrays with panel-row-level information: 19227
Total area of GM-SEUS arrays: 1131.45 km2


## Merge New Arrays with Existing Array Dataset to Create the Highest Quality Array Shapefile
* We replace TZSAM, OSM, GRW, and CWSD array boundaries with new array boundaries where detected. These array shapes either do not conform with our array definition, were derived by low spatial resolution methods creating problematic array bounds, or do not have have a standardized delineation methods.
* This is what is uploaded to GEE asset and used in script5. 

In [11]:
# Call GMSEUS arrays init and NAIP Arrays from panels
gmseusArraysInit = gpd.read_file(gmseusArraysInitPath)
gmseusArraysFromPanels = gpd.read_file(os.path.join(derivedTemp_path, r'GMSEUS_ArraysFromPanels.shp'))

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Select when to merge GM-SEUS arrays with existing arrays to create the highest quality array dataset

'''
In gmseusArraysInit, if Source is TZSAM, CWSD, GRW, or OSM, replace the array geometry with the gmseusArraysFromPanels geometry that intersects that array. 
If not, retain geometries. Set newBound to 1 if a geometry is replaced.
Additionally, if source is TZSAM, GRW, or CWSD, explode the arrays since SAM and CWSD does not necessarily have project level infomration, and merges many array boundaries into one. 
'''

# Add a replaceID column to gmseusArraysFromPanels that is 1
gmseusArraysFromPanels['replaceID'] = 1

# Set a new column to gmseusArraysInit that is newBound, which will be a binary indicator for if a new array boundary is used. 
gmseusArraysInit['newBound'] = 0

# ~~~~~~~~~~~~~ Replace TZSAM, CWSD, GRW, and OSM arrays with GMSEUSnaip arrays

# Filter for TZSAM, CWSD, GRW, and OSM sources
replaceBoundaries = gmseusArraysInit[gmseusArraysInit['Source'].isin(['TZSAM', 'OSM', 'GRW', 'CWSD'])]

# Perform a spatial join to find intersecting geometries from gmseusArraysFromPanels
replaceBoundaries_joined = gpd.sjoin(replaceBoundaries, gmseusArraysFromPanels[['geometry', 'replaceID']], how='right', predicate='intersects') # KEY: Take geometry from right side of join

# Group by arrayID to get all intersecting geometries, and drop rows where replaceID is null
replaceBoundaries_joined = replaceBoundaries_joined.groupby(arrayIDcol).first().reset_index()
replaceBoundaries_joined = replaceBoundaries_joined.dropna(subset=['replaceID']).reset_index()

# If gmseusArraysInit arrayID is in replaceBoundaries_joined arrayID, set the geometry to the geometry in replaceBoundaries_joined and set newBound to 1
gmseusArraysInit = gmseusArraysInit.merge(replaceBoundaries_joined[[arrayIDcol, 'geometry']], on=arrayIDcol, how='left')
gmseusArraysInit['geometry'] = np.where(gmseusArraysInit['geometry_y'].isnull(), gmseusArraysInit['geometry_x'], gmseusArraysInit['geometry_y'])
gmseusArraysInit['newBound'] = np.where(gmseusArraysInit['geometry_y'].isnull(), gmseusArraysInit['newBound'], 1)
gmseusArraysInit = gmseusArraysInit.drop(columns=['geometry_x', 'geometry_y'])

# Save as geodataframe
gmseusArraysInit = gpd.GeoDataFrame(gmseusArraysInit, crs=toCRS)

# ~~~~~~~~~~~~~ For TZSAM arrays, explode the arrays to get individual array boundaries because TZSAM does not contain project level information

# We do this because TZSAM has a tendency to identify multiple arrays as one array. Qualitative, this is only the case for TZSAM, while GRW and CWSD tend to underrepsent array area for newly generated bounds and OSM is hand delineated meaning arrays are distinct.

# Filter for TZSAM sources
tzsam = gmseusArraysInit[gmseusArraysInit['Source'] == 'TZSAM']

# Explode the arrays
tzsam_exploded = tzsam.explode(index_parts=False)

# Drop TZSAM arrays from gmseusArraysInit
gmseusArraysInit = gmseusArraysInit[gmseusArraysInit['Source'] != 'TZSAM']

# Concatenate the exploded TZSAM arrays with gmseusArraysInit
gmseusArraysInit = pd.concat([gmseusArraysInit, tzsam_exploded], ignore_index=True)

# ~~~~~~~~~~~~~ Save and export the final GM-SEUS arrays and print some statistics

# Save copy of gmseusArraysInit as gmseusArraysFinal
gmseusArraysFinal = gmseusArraysInit.copy()

# Drop arrayIDcol and Subset column
gmseusArraysFinal = gmseusArraysFinal.drop(columns=[arrayIDcol, 'Subset'], errors='ignore')

# Add a temporary ID column that is 1 through n for the exploded arrays
gmseusArraysFinal[arrayIDcol] = range(1, len(gmseusArraysFinal) + 1)

# Count the number of arrays with new boundaries
newBoundCount = len(gmseusArraysFinal[gmseusArraysFinal['newBound'] == 1])

# Print the number of arrays with new boundaries
print(f'Preliminary total number of arrays with new boundaries: {newBoundCount}')

# Print the total number of arrays in gmseusArraysFinal
print(f'Preliminary total number of arrays in GM-SEUS arrays: {len(gmseusArraysFinal)}')

# Calculate and print the total area of gmseusArraysFinal
print(f'Preliminary total area of GM-SEUS arrays: {gmseusArraysFinal.geometry.area.sum() / 1e6:.2f} km2')

# Export gmseusArraysFinal
gmseusArraysFinal.to_file(os.path.join(derivedTemp_path, r'GMSEUS_Arrays_selectedBoundaries.shp'), driver='ESRI Shapefile')

Preliminary total number of arrays with new boundaries: 8849
Preliminary total number of arrays in GM-SEUS arrays: 31306
Preliminary total area of GM-SEUS arrays: 3846.85 km2
